# Chapter 12 &mdash; Stack Limits and Loop Prevention

**Concept 7 of the Chapter 12 decomposition:** *PDA Behavior Through Examples, Stack Limits, and Loop Prevention*

An $\varepsilon$-loop can push forever; Jove prunes an ID whose stack has grown by `STKMAX` at a configuration it has already seen.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12/Concept-Stack-Limits-And-Loops/Concept-Stack-Limits-And-Loops.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A PDA can **loop forever without reading input**: `q : '' , X ; XX -> q` pushes on
every $\varepsilon$ move and never terminates. Exhaustive simulation of such a machine
would not halt.

Jove's cure is **subsumption**, controlled by `STKMAX`. An ID is discarded when a
previously visited ID has the **same state and the same remaining input** but a stack
shorter by at least `STKMAX` symbols. In other words: *you have been here before,
only with less on the stack, and piling more on is not helping.*

Read the consequence carefully &mdash; it is **not** an absolute depth cap. Deeply nested
input is fine, because each step consumes a symbol and so reaches a *different*
configuration. What gets cut is growth **at a configuration already seen**, which is
exactly the signature of an $\varepsilon$-loop.

## 2. Definitions

### A machine with an epsilon-loop

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
Looper = md2mc('''PDA
I : '' , # ; X#  -> I     !! pushes forever, reading nothing
I : '' , X ; XX  -> I
I : a , X ; ''   -> I
I : '' , # ; #   -> F
''')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### A well-behaved machine that simply nests deeply

In [ ]:
Deep = md2mc('''PDA
I : ( , # ; (#  -> I
I : ( , ( ; ((  -> I
I : ) , ( ; ''  -> I
I : '' , # ; #  -> F
''')

## 3. Tests

`STKMAX` keeps the search finite even with an $\varepsilon$-loop.

In [ ]:
for k in [2, 4, 6, 8]:
    surv, paths, visited = run_pda('a', Looper, STKMAX=k)
    deepest = max(len(st) for (_, _, st) in visited)
    print("  STKMAX=%d : %3d IDs explored, deepest stack %2d, accepted? %s"
          % (k, len(visited), deepest, bool(paths)))
print("\nthe exploration grows with STKMAX and stops; without it, it would not.")

**It is not an absolute depth cap.** Deep nesting is unaffected.

In [ ]:
for n in range(1, 7):
    s = '(' * n + ')' * n
    got = [pda_accepts(Deep, s, STKMAX=k) for k in (1, 2, 6)]
    print("  depth %d : accepted at STKMAX 1, 2, 6 -> %s" % (n, got))
    assert all(got)
print("\nEach input symbol changes the remaining input, so no two IDs on this")
print("run share a configuration -- subsumption never fires.")

What subsumption actually compares.

In [ ]:
print("an ID is (state, remaining input, stack)")
print()
print("discard ID2 when some visited ID1 has")
print("   same state, same remaining input, and  |stk2| - |stk1| >= STKMAX")
print()
print("Same state and same remaining input means the machine is in the SAME")
print("situation.  A taller stack there is progress only an epsilon-loop makes.")

The loop is visible: it reaches the same configuration with ever more on the stack.

In [ ]:
surv, paths, visited = run_pda('a', Looper, STKMAX=6)
same_config = sorted({st for (q, inp, st) in visited if q == 'I' and inp == 'a'})
print("stacks seen at (I, 'a') :", same_config)
assert len(same_config) > 1
print("\nsame state, same remaining input, growing stack -- that is the loop.")

Spotting the danger statically: an $\varepsilon$ move that pushes more than it pops.

In [ ]:
def eps_growers(P):
    return [(k, sorted(v)) for k, v in P["Delta"].items()
            if k[1] == '' and any(len(push) > 1 for (_, push) in v)]
print("Looper :", eps_growers(Looper))
print("Deep   :", eps_growers(Deep))
assert eps_growers(Looper) and not eps_growers(Deep)
print("\nAn epsilon edge whose push-string is longer than its pop is the danger sign.")

## 4. Exercises


1. Build a PDA with an $\varepsilon$-loop that *pops* forever. Does subsumption help?
2. Why must subsumption compare the **remaining input** and not just the state?
3. Why can a DFA simulator never need an analogue of `STKMAX`?

In [ ]:
# Your work for the exercises above.